In [1]:
import os
import sys
import pandas as pd
from pathlib import Path

# Ensure project root is on sys.path so 'backtester' imports work
cwd = Path(os.getcwd())
candidates = [cwd, cwd.parent, Path("..").resolve()]
for p in candidates:
    if (p / "backtester").exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        break

print("PYTHONPATH set. Using root:", sys.path[0])

PYTHONPATH set. Using root: c:\Users\User\Desktop\Crypto strategy backtest


In [2]:
def _make_df(bo_rows: int = 10, pd_rows = 3, second_leg_rows = 10, side: str = "long") -> pd.DataFrame:
    idx = pd.date_range("2026-01-01", periods=bo_rows+pd_rows+second_leg_rows, freq="5min")
    if side == "long":
        df = pd.DataFrame([])
        df_BO = pd.DataFrame(
            {
                "open":  [100.0 + i for i in range(bo_rows)],
                "high":  [101.0 + i for i in range(bo_rows)],
                "low":   [99.0 + i for i in range(bo_rows)],
                "close": [100.5 + i for i in range(bo_rows)],
            },
            index=idx[:bo_rows],
        )
        BO_last_close = df_BO["close"].iat[-1]
        df_PB = pd.DataFrame(
            {
                "open":  [BO_last_close - i for i in range(pd_rows)],
                "high":  [BO_last_close + 1 - i for i in range(pd_rows)],
                "low":   [BO_last_close - 1 - i for i in range(pd_rows)],
                "close": [BO_last_close - 0.5 - i for i in range(pd_rows)],
            },
            index=idx[bo_rows:bo_rows+pd_rows],
        )
        pb_last_close = df_PB["close"].iat[-1]
        df_second_leg = pd.DataFrame(
            {
                "open":  [pb_last_close + i for i in range(second_leg_rows)],
                "high":  [pb_last_close + 1 + i for i in range(second_leg_rows)],
                "low":   [pb_last_close - 1 + i for i in range(second_leg_rows)],
                "close": [pb_last_close + 0.5 + i for i in range(second_leg_rows)],
            },
            index=idx[bo_rows+pd_rows:bo_rows+pd_rows+second_leg_rows],
        )
        df = pd.concat([df_BO, df_PB, df_second_leg], ignore_index=True)
        return df
    else:
        df = pd.DataFrame([])
        df_BO = pd.DataFrame(
            {
                "open":  [100.0 - i for i in range(bo_rows)],
                "high":  [101.0 - i for i in range(bo_rows)],
                "low":   [99.0 - i for i in range(bo_rows)],
                "close": [99.5 - i for i in range(bo_rows)],
            },
            index=idx[:bo_rows],
        )
        BO_last_close = df_BO["close"].iat[-1]
        df_PB = pd.DataFrame(
            {
                "open":  [BO_last_close + i for i in range(pd_rows)],
                "high":  [BO_last_close + 1 + i for i in range(pd_rows)],
                "low":   [BO_last_close - 1 + i for i in range(pd_rows)],
                "close": [BO_last_close + 0.5 + i for i in range(pd_rows)],
            },
            index=idx[bo_rows:bo_rows+pd_rows],
        )
        pb_last_close = df_PB["close"].iat[-1]
        df_second_leg = pd.DataFrame(
            {
                "open":  [pb_last_close - i for i in range(second_leg_rows)],
                "high":  [pb_last_close + 1 - i for i in range(second_leg_rows)],
                "low":   [pb_last_close - 1 - i for i in range(second_leg_rows)],
                "close": [pb_last_close - 0.5 - i for i in range(second_leg_rows)],
            },
            index=idx[bo_rows+pd_rows:bo_rows+pd_rows+second_leg_rows],
        )
        df = pd.concat([df_BO, df_PB, df_second_leg], ignore_index=True)
        return df

In [3]:

def load_crypto_parquet_data(coin_name: str, timeframe: str = "5m", nM: int = 54, section: str = "UTC") -> pd.DataFrame:
    df = pd.read_parquet(fr'C:\Users\User\Desktop\Crypto\{coin_name}_{timeframe}_{nM}M_{section}.parquet')
    return df
def generate_us_session_bars_info(df, include_holidays: bool = False):
    # 確保時間有時區資訊
    df['dt_ny'] = pd.to_datetime(df['dt_utc'], utc=True).dt.tz_convert('America/New_York')

    # 取日期（當地日曆）
    df['date'] = df['dt_ny'].dt.date
    df['weekday'] = df['dt_ny'].dt.day_name() 
    # 對每天依時間排序並編號
    df = df.sort_values(['date', 'dt_ny']).reset_index(drop=True)
    df['bar_index'] = df.groupby('date').cumcount() + 1  # 第幾根K線，從1開始
    if not include_holidays:
        weekday = ['Monday','Tuesday','Wednesday','Thursday','Friday']
    else:
        weekday = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    df = df.loc[df['weekday'].isin(weekday)]
    # 查看結果
    df.set_index('dt_utc', inplace=True)
    # print(df[['dt_ny', 'date', 'weekday', 'bar_index']].head(5))

    return df
def generate_allday_bars_info(df, include_holidays: bool = True):
# 確保時間有時區資訊
    df['dt_ny'] = pd.to_datetime(df['dt_utc'], utc=True).dt.tz_convert('America/New_York')

    # 取日期（當地日曆）
    df['date'] = df['dt_ny'].dt.date
    df['time'] = df['dt_ny'].dt.time
    df['weekday'] = df['dt_ny'].dt.day_name() 
    # 對每天依時間排序並編號
    df = df.sort_values(['date', 'dt_ny']).reset_index(drop=True)
    df['bar_index'] = df.groupby('date').cumcount() + 1  # 第幾根K線，從1開始
    if not include_holidays:
        weekday = ['Monday','Tuesday','Wednesday','Thursday','Friday']
    else:
        weekday = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    df = df.loc[df['weekday'].isin(weekday)]
    # 查看結果
    df.set_index('dt_utc', inplace=True)
    # print(df[['dt_ny', 'date', 'weekday', 'bar_index']].head(5))

    return df


In [ ]:
import backtester.indicators as idc

df = _make_df(bo_rows=10, pd_rows=3, second_leg_rows=10, side="long")
df["ll_streak"] = idc.hh_ll_streak(df, side="ll")
df["hh_streak"] = idc.hh_ll_streak(df, side="hh")
df["ll_check"] = idc.hh_ll_check(df, side="ll")
df["hh_check"] = idc.hh_ll_check(df, side="hh")
df["pivot_low_mask"] = idc.pivot_mask(df, "low", 3)

df

,open,high,low,close,ll_check,hh_check,pivot_low_mask
0,100.0,101.0,99.0,100.5,0,0,False
1,101.0,102.0,100.0,101.5,0,1,False
2,102.0,103.0,101.0,102.5,0,2,False
3,103.0,104.0,102.0,103.5,0,3,False
4,104.0,105.0,103.0,104.5,0,4,False
5,105.0,106.0,104.0,105.5,0,5,False
6,106.0,107.0,105.0,106.5,0,6,False
7,107.0,108.0,106.0,107.5,0,7,False
8,108.0,109.0,107.0,108.5,0,8,False
9,109.0,110.0,108.0,109.5,0,9,False
